In [1]:
# ============================================================
# 🎬 Netflix Recommendation Model — TFRS (PRODUCTION READY)
# Retrieval • Embeddings • InferStream-compatible
# ============================================================

import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow_recommenders as tfrs
import os
import json
from datetime import datetime

# ================================
# Paths
# ================================
DATA_PATH = "../data/netflix_customer_churn.csv"
MODEL_DIR = "../backend/models/netflix"

MODEL_WEIGHTS = "tfrs_model.weights.h5"
USER_VOCAB = "tfrs_user_vocab.json"
GENRE_VOCAB = "tfrs_genre_vocab.json"
MODEL_META = "tfrs_model_meta.json"

os.makedirs(MODEL_DIR, exist_ok=True)

# ================================
# Load dataset
# ================================
df = pd.read_csv(DATA_PATH)
print("✅ Loaded data:", df.shape)

rec_df = df[["customer_id", "favorite_genre", "avg_watch_time_per_day"]].dropna()
rec_df["customer_id"] = rec_df["customer_id"].astype(str)
rec_df["favorite_genre"] = rec_df["favorite_genre"].astype(str)

# ================================
# Normalize watch time → rating
# ================================
def normalize_watch_time(x):
    return max(1, min(5, round(x / 60 * 5)))

rec_df["rating"] = rec_df["avg_watch_time_per_day"].apply(normalize_watch_time)

# ================================
# Vocabularies (FROZEN)
# ================================
user_ids = sorted(rec_df["customer_id"].unique().tolist())
genres = sorted(rec_df["favorite_genre"].unique().tolist())

# Save vocabularies (CRITICAL)
with open(os.path.join(MODEL_DIR, USER_VOCAB), "w") as f:
    json.dump(user_ids, f, indent=2)

with open(os.path.join(MODEL_DIR, GENRE_VOCAB), "w") as f:
    json.dump(genres, f, indent=2)

# ================================
# Dataset
# ================================
ratings = tf.data.Dataset.from_tensor_slices({
    "customer_id": rec_df["customer_id"].values,
    "favorite_genre": rec_df["favorite_genre"].values,
    "rating": rec_df["rating"].astype(np.float32).values,
})

ratings = ratings.shuffle(10_000).batch(256).cache()

# ================================
# Models
# ================================
EMBED_DIM = 32

class UserModel(tf.keras.Model):
    def __init__(self, vocab):
        super().__init__()
        self.lookup = tf.keras.layers.StringLookup(
            vocabulary=vocab, mask_token=None
        )
        self.embed = tf.keras.layers.Embedding(len(vocab) + 1, EMBED_DIM)

    def call(self, inputs):
        return self.embed(self.lookup(inputs))

class GenreModel(tf.keras.Model):
    def __init__(self, vocab):
        super().__init__()
        self.lookup = tf.keras.layers.StringLookup(
            vocabulary=vocab, mask_token=None
        )
        self.embed = tf.keras.layers.Embedding(len(vocab) + 1, EMBED_DIM)

    def call(self, inputs):
        return self.embed(self.lookup(inputs))

class NetflixRetrievalModel(tfrs.models.Model):
    def __init__(self, user_model, genre_model):
        super().__init__()
        self.user_model = user_model
        self.genre_model = genre_model
        self.task = tfrs.tasks.Retrieval(
            loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True)
        )

    def compute_loss(self, features, training=False):
        user_embeddings = self.user_model(features["customer_id"])
        genre_embeddings = self.genre_model(features["favorite_genre"])
        return self.task(user_embeddings, genre_embeddings)

# ================================
# Train
# ================================
user_model = UserModel(user_ids)
genre_model = GenreModel(genres)
model = NetflixRetrievalModel(user_model, genre_model)

model.compile(optimizer=tf.keras.optimizers.Adagrad(0.5))
model.fit(ratings, epochs=5)

# Explicit build (prevents TF bug)
model.build({
    "customer_id": tf.TensorShape([None]),
    "favorite_genre": tf.TensorShape([None]),
    "rating": tf.TensorShape([None]),
})

# ================================
# Save model weights
# ================================
model.save_weights(os.path.join(MODEL_DIR, MODEL_WEIGHTS))

# Save metadata
with open(os.path.join(MODEL_DIR, MODEL_META), "w") as f:
    json.dump(
        {
            "embedding_dim": EMBED_DIM,
            "model_type": "tfrs_retrieval",
            "user_vocab_size": len(user_ids),
            "genre_vocab_size": len(genres),
        },
        f,
        indent=2,
    )

print("✅ Saved artifacts:")
print(f"   • {MODEL_WEIGHTS}")
print(f"   • {USER_VOCAB}")
print(f"   • {GENRE_VOCAB}")
print(f"   • {MODEL_META}")

print("🏁 Done at", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

✅ Loaded data: (5000, 14)
Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 4.9125 - regularization_loss: 0.0000e+00 - total_loss: 4.9125  
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 4.9120 - regularization_loss: 0.0000e+00 - total_loss: 4.9120
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 4.9114 - regularization_loss: 0.0000e+00 - total_loss: 4.9114
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 4.9106 - regularization_loss: 0.0000e+00 - total_loss: 4.9106
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.9095 - regularization_loss: 0.0000e+00 - total_loss: 4.9095
✅ Saved artifacts:
   • tfrs_model.weights.h5
   • tfrs_user_vocab.json
   • tfrs_genre_vocab.json
   • tfrs_model_meta.json
🏁 Done at 2026-01-24 18:51:38
